# NB_LIBRO_10 · Figuras y experimentos del capítulo 8

**Libro:** *AgroVisión: del píxel a la decisión* · capítulo 8, «Caso avícola: detección y conteo»

Este notebook:

1. aplica el detector YOLO11s del proyecto a las 452 imágenes de validación del conjunto PIO y guarda todas las detecciones, emparejadas con las anotaciones;
2. repite la inferencia con tres tamaños de imagen (640, 960 y 1280 píxeles) y tres límites de supresión de no máximos;
3. dibuja una escena con sus aciertos, sus detecciones sin anotación y sus aves no detectadas;
4. reúne en dos galerías los errores más claros de cada tipo;
5. hace un experimento nuevo: entrena el mismo detector con imágenes de 640 píxeles para medir cuánto aporta la resolución.

**Usa (solo lectura):** `AgroUTMACH_PIX/01_DATASET_PIO` y `03_MODELOS/AgroUTMACH_PIX_YOLO11s_PIO_best.pt`. No modifica nada del proyecto.

**Produce:** dos figuras, `valores_cap08.json`, las detecciones en `cap08_detecciones.npz` y los resultados del entrenamiento a 640 píxeles, en `AGROVISION_LIBRO`. Al final descarga un zip.

**Cómo ejecutarlo:** *Entorno de ejecución → Cambiar tipo de entorno → GPU* (L4 recomendada) y *Ejecutar todo*. Tarda de 35 a 45 minutos; la mayor parte es el entrenamiento del final. Si Colab se desconecta, vuelva a ejecutar todo: el entrenamiento terminado se reutiliza. Suba al chat el zip que se descarga al final.

In [ ]:
%pip install -q ultralytics==8.4.172

In [ ]:
import os, re, json, math, zipfile, time, shutil, gc, random, traceback
from pathlib import Path

try:
    from google.colab import drive
    drive.mount('/content/drive')
    EN_COLAB = True
except ImportError:
    EN_COLAB = False

BASE = Path(os.environ.get('AGROVISION_BASE', '/content/drive/MyDrive'))
LOCAL = Path(os.environ.get('AGROVISION_LOCAL', '/content'))
PIX = BASE / 'AgroUTMACH_PIX'
PIO = PIX / '01_DATASET_PIO' / 'PIO_TRAIN_VAL' / 'data'
MOD_PIO = PIX / '03_MODELOS' / 'AgroUTMACH_PIX_YOLO11s_PIO_best.pt'
DIR_FIG = BASE / 'AGROVISION_LIBRO' / '02_FIGURAS' / 'cap08'
DIR_VAL = BASE / 'AGROVISION_LIBRO' / '03_VALORES'
DIR_EXP = DIR_VAL / 'cap08_640'
for d in (DIR_FIG, DIR_VAL, DIR_EXP):
    d.mkdir(parents=True, exist_ok=True)
for p in (PIO / 'images' / 'val', PIO / 'labels' / 'val', MOD_PIO):
    print('OK   ' if p.exists() else 'FALTA', p)

ENTRENAR_640 = True          # ponga False para omitir el entrenamiento a 640 píxeles (unos 20 minutos)
CONF_CONTEO = 0.25           # umbral de confianza con el que el proyecto cuenta

import torch, ultralytics
from ultralytics import YOLO
HAY_GPU = torch.cuda.is_available()
DISP = 0 if HAY_GPU else 'cpu'
GPU = torch.cuda.get_device_name(0) if HAY_GPU else 'sin GPU'
print('GPU:', GPU, '| PyTorch', torch.__version__, '| Ultralytics', ultralytics.__version__)
assert HAY_GPU or not EN_COLAB, 'Active una GPU en Entorno de ejecución > Cambiar tipo de entorno.'

# Copia local del conjunto: evita leer de Drive en cada pasada. El archivo classes.txt no es una anotación.
DATOS = LOCAL / 'PIO_cap08'
if not (DATOS / 'listo.txt').exists():
    if DATOS.exists():
        shutil.rmtree(DATOS)
    t0 = time.time()
    shutil.copytree(PIO / 'images', DATOS / 'images'); shutil.copytree(PIO / 'labels', DATOS / 'labels')
    for f in DATOS.rglob('classes.txt'):
        f.unlink()
    (DATOS / 'listo.txt').write_text('ok')
    print(f'Copia local en {(time.time() - t0) / 60:.1f} min')
YAML = DATOS / 'pio_cap08.yaml'
YAML.write_text(f'path: {DATOS}\ntrain: images/train\nval: images/val\nnc: 1\nnames:\n- Pollo\n', encoding='utf-8')

In [ ]:
import numpy as np
import pandas as pd
import cv2
import matplotlib
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap
from matplotlib.patches import FancyBboxPatch, Rectangle
from matplotlib.ticker import FuncFormatter
from PIL import Image, ImageOps
from skimage.color import rgb2lab, lab2rgb, rgb2hsv

ANCHO_CM = 16.0                      # ancho de caja de texto del libro
ANCHO_IN = ANCHO_CM / 2.54
DPI = 300

# Tinta y colores de apoyo. Los datos categóricos no se distinguen solo por color:
# las cinco clases se muestran en paneles separados (Figura 2.7).
TINTA = '#0b0b0b'
TINTA_2 = '#52514e'
GRIS_FONDO = '#c9c8c2'
ACENTO = '#2a78d6'
REJILLA = '#e4e3de'

plt.rcParams.update({
    'font.family': 'DejaVu Sans',
    'font.size': 8,
    'axes.titlesize': 8,
    'axes.labelsize': 8,
    'xtick.labelsize': 7,
    'ytick.labelsize': 7,
    'legend.fontsize': 7,
    'axes.edgecolor': TINTA_2,
    'axes.labelcolor': TINTA,
    'text.color': TINTA,
    'xtick.color': TINTA_2,
    'ytick.color': TINTA_2,
    'axes.linewidth': 0.6,
    'axes.spines.top': False,
    'axes.spines.right': False,
    'figure.dpi': 110,
    'savefig.dpi': DPI,
    'figure.facecolor': 'white',
    'savefig.facecolor': 'white',
})

CLASES = ['unripe', 'ripe', 'overripe', 'damaged', 'empty']       # orden de presentación
NOMBRE_ES = {'unripe': 'Inmaduro', 'ripe': 'Maduro', 'overripe': 'Sobremaduro',
             'damaged': 'Dañado', 'empty': 'Vacío'}
EXT_IMG = {'.jpg', '.jpeg', '.png', '.bmp', '.tif', '.tiff', '.webp'}

VALORES = {}          # cifras que se copian al manuscrito
FIGURAS = []          # rutas de las figuras generadas


def guardar(fig, ruta):
    # Guarda la figura a 300 dpi y la registra.
    fig.savefig(ruta, dpi=DPI, bbox_inches='tight', pad_inches=0.04)
    FIGURAS.append(str(ruta))
    print('Guardada:', ruta.name)


def leer_rgb(ruta, lado_max=None):
    # Lee una imagen como arreglo RGB uint8, respetando la orientación EXIF.
    # Con lado_max, la reduce para que su lado mayor no supere ese valor.
    with Image.open(ruta) as im:
        if lado_max is not None and im.format == 'JPEG':
            im.draft('RGB', (lado_max * 2, lado_max * 2))     # decodificación rápida
        im = ImageOps.exif_transpose(im).convert('RGB')
        if lado_max is not None and max(im.size) > lado_max:
            im.thumbnail((lado_max, lado_max), Image.LANCZOS)
        return np.asarray(im)


def redimensionar(img, lado_mayor):
    # Redimensiona conservando la proporción hasta que el lado mayor mida lado_mayor.
    h, w = img.shape[:2]
    escala = lado_mayor / max(h, w)
    nuevo = (max(1, round(w * escala)), max(1, round(h * escala)))
    interp = cv2.INTER_AREA if escala < 1 else cv2.INTER_CUBIC
    return cv2.resize(img, nuevo, interpolation=interp)


def recorte_cuadrado(img, lado=512):
    # Recorte cuadrado centrado, redimensionado a lado × lado píxeles.
    h, w = img.shape[:2]
    m = min(h, w)
    y0, x0 = (h - m) // 2, (w - m) // 2
    return cv2.resize(img[y0:y0 + m, x0:x0 + m], (lado, lado), interpolation=cv2.INTER_AREA)


def coma(x, dec=1):
    # Número con coma decimal, como se escribe en español.
    return f'{x:.{dec}f}'.replace('.', ',')


def eje_con_coma(eje, dec=2):
    eje.set_major_formatter(FuncFormatter(lambda v, _: coma(v, dec)))


def sin_ejes(ax):
    ax.set_xticks([]); ax.set_yticks([])
    for s in ax.spines.values():
        s.set_visible(False)


print('Bibliotecas listas. matplotlib', matplotlib.__version__, '| OpenCV', cv2.__version__)


ERRORES = {}
C_FP, C_FN = '#e8590c', '#ffd43b'          # detección sin anotación · ave anotada no detectada
RUTAS = sorted(p for p in (DATOS / 'images' / 'val').iterdir() if p.suffix.lower() in EXT_IMG)
print('Imágenes de validación:', len(RUTAS))
REF = None


def seccion(nombre, fn):
    # Ejecuta una sección y, si falla, anota el error y deja que las demás continúen.
    t0 = time.time()
    try:
        fn()
        print(f'[{nombre}] terminada en {time.time() - t0:.0f} s')
    except Exception:
        ERRORES[nombre] = traceback.format_exc()[-1500:]
        print(f'[{nombre}] ERROR\n{ERRORES[nombre]}')
    plt.close('all')


def iou_matriz(a, b):
    # IoU entre cada caja de a (n × 4) y cada caja de b (m × 4), en formato x1, y1, x2, y2.
    if len(a) == 0 or len(b) == 0:
        return np.zeros((len(a), len(b)), np.float32)
    x1 = np.maximum(a[:, None, 0], b[None, :, 0]); y1 = np.maximum(a[:, None, 1], b[None, :, 1])
    x2 = np.minimum(a[:, None, 2], b[None, :, 2]); y2 = np.minimum(a[:, None, 3], b[None, :, 3])
    inter = np.clip(x2 - x1, 0, None) * np.clip(y2 - y1, 0, None)
    area_a = (a[:, 2] - a[:, 0]) * (a[:, 3] - a[:, 1]); area_b = (b[:, 2] - b[:, 0]) * (b[:, 3] - b[:, 1])
    return (inter / (area_a[:, None] + area_b[None, :] - inter + 1e-12)).astype(np.float32)


def leer_cajas(ruta_txt, ancho=1.0, alto=1.0):
    # Lee un archivo YOLO y devuelve las cajas como x1, y1, x2, y2; descarta las de ancho o alto nulo.
    filas = [l.split() for l in Path(ruta_txt).read_text(encoding='utf-8', errors='ignore').splitlines() if len(l.split()) == 5]
    if not filas:
        return np.zeros((0, 4), np.float32)
    a = np.array(filas, dtype=np.float32)[:, 1:]
    c = np.column_stack([(a[:, 0] - a[:, 2] / 2) * ancho, (a[:, 1] - a[:, 3] / 2) * alto, (a[:, 0] + a[:, 2] / 2) * ancho, (a[:, 1] + a[:, 3] / 2) * alto])
    return c[(c[:, 2] > c[:, 0]) & (c[:, 3] > c[:, 1])]


def emparejar(M, conf, umbral):
    # Emparejamiento voraz: de mayor a menor confianza, cada detección toma la anotación libre con mayor IoU.
    # Devuelve, para cada detección, el índice de su anotación o -1.
    par = np.full(len(conf), -1, np.int32); libre = np.ones(M.shape[1], bool)
    if M.shape[1] == 0:
        return par
    for k in np.argsort(-conf):
        fila = np.where(libre, M[k], -1.0); j = int(fila.argmax())
        if fila[j] >= umbral:
            par[k] = j; libre[j] = False
    return par


def precision_media(conf, tp, n_gt):
    # Precisión media con interpolación de 101 puntos, como en COCO.
    orden = np.argsort(-conf); tp = tp[orden]
    acum_tp = np.cumsum(tp); acum_fp = np.cumsum(~tp)
    sens = acum_tp / max(n_gt, 1); prec = acum_tp / np.maximum(acum_tp + acum_fp, 1)
    env = np.maximum.accumulate(prec[::-1])[::-1]
    r = np.linspace(0, 1, 101); idx = np.searchsorted(sens, r, side='left')
    return float(np.where(idx < len(env), env[np.minimum(idx, len(env) - 1)], 0).mean())


UMBRALES_IOU = np.round(np.arange(0.5, 0.96, 0.05), 2)


def pasada(modelo, lado, iou_nms):
    # Inferencia sobre las 452 imágenes de validación. Devuelve una lista con cajas, confianzas y anotaciones.
    reg, ms = [], []
    for k in range(0, len(RUTAS), 16):
        lote = RUTAS[k:k + 16]
        res = modelo.predict(source=[str(p) for p in lote], imgsz=lado, conf=0.001, iou=iou_nms, max_det=1000, device=DISP, verbose=False)
        for ruta, r in zip(lote, res):
            alto, ancho = r.orig_shape
            cajas = r.boxes.xyxy.detach().cpu().numpy().astype(np.float32); conf = r.boxes.conf.detach().cpu().numpy().astype(np.float32)
            gt = leer_cajas(DATOS / 'labels' / 'val' / (ruta.stem + '.txt'), ancho, alto)
            reg.append({'archivo': ruta.name, 'ancho': int(ancho), 'alto': int(alto), 'cajas': cajas, 'conf': conf, 'gt': gt, 'M': iou_matriz(cajas, gt)})
            v = getattr(r, 'speed', None)
            if v:
                ms.append(float(sum(v.values())))
    return reg, (float(np.mean(ms)) if ms else None)


def resumir(reg, ms=None):
    conf = np.concatenate([x['conf'] for x in reg]); n_gt = int(sum(len(x['gt']) for x in reg))
    ap = {}
    for u in UMBRALES_IOU:
        tp = np.concatenate([emparejar(x['M'], x['conf'], u) >= 0 for x in reg]); ap[float(u)] = precision_media(conf, tp, n_gt)
    tp50 = np.concatenate([emparejar(x['M'], x['conf'], 0.5) >= 0 for x in reg]); m = conf >= CONF_CONTEO
    dif = np.array([int((x['conf'] >= CONF_CONTEO).sum()) - len(x['gt']) for x in reg])
    return {'AP50': ap[0.5], 'AP75': ap[0.75], 'AP50_95': float(np.mean(list(ap.values()))), 'AP_por_umbral': ap,
            'detecciones': int(len(conf)), 'anotaciones': n_gt, 'precision': float(tp50[m].mean()), 'sensibilidad': float(tp50[m].sum() / n_gt),
            'conteo_eam': float(np.abs(dif).mean()), 'conteo_sesgo': float(dif.mean()), 'ms_por_imagen': ms}

## 1. El detector del proyecto, caja por caja

In [ ]:
def s_referencia():
    global REF
    det = YOLO(str(MOD_PIO))
    REF, ms = pasada(det, 960, 0.7)
    VALORES['referencia'] = resumir(REF, ms)
    print({k: (round(v, 4) if isinstance(v, float) else v) for k, v in VALORES['referencia'].items() if k != 'AP_por_umbral'})
    # Todas las detecciones y todas las anotaciones, con su emparejamiento a IoU 0,5, para analizarlas después.
    d_img, d_caja, d_conf, d_iou, d_par, g_img, g_caja, g_conf = [], [], [], [], [], [], [], []
    for i, x in enumerate(REF):
        par = emparejar(x['M'], x['conf'], 0.5); x['par'] = par
        conf_gt = np.zeros(len(x['gt']), np.float32); conf_gt[par[par >= 0]] = x['conf'][par >= 0]; x['conf_gt'] = conf_gt
        d_img.append(np.full(len(par), i, np.int16)); d_caja.append(x['cajas']); d_conf.append(x['conf']); d_par.append(par)
        d_iou.append(x['M'].max(1) if x['M'].shape[1] else np.zeros(len(par), np.float32))
        g_img.append(np.full(len(x['gt']), i, np.int16)); g_caja.append(x['gt']); g_conf.append(conf_gt)
    np.savez_compressed(DIR_VAL / 'cap08_detecciones.npz', d_img=np.concatenate(d_img), d_caja=np.concatenate(d_caja).round(1),
                        d_conf=np.concatenate(d_conf), d_iou=np.concatenate(d_iou), d_par=np.concatenate(d_par),
                        g_img=np.concatenate(g_img), g_caja=np.concatenate(g_caja).round(1), g_conf=np.concatenate(g_conf))
    pd.DataFrame({'archivo': [x['archivo'] for x in REF], 'ancho': [x['ancho'] for x in REF], 'alto': [x['alto'] for x in REF],
                  'anotadas': [len(x['gt']) for x in REF], 'detectadas': [int((x['conf'] >= CONF_CONTEO).sum()) for x in REF]}
                 ).to_csv(DIR_VAL / 'cap08_imagenes.csv', index=False)

seccion('referencia', s_referencia)

## 2. Tamaño de imagen y supresión de no máximos

In [ ]:
def s_barrido():
    det = YOLO(str(MOD_PIO)); filas = []
    for lado in (640, 960, 1280):
        for iou_nms in (0.5, 0.6, 0.7):
            reg, ms = pasada(det, lado, iou_nms)
            r = resumir(reg, ms); r.pop('AP_por_umbral'); r.update({'lado': lado, 'iou_nms': iou_nms}); filas.append(r)
            print(lado, iou_nms, {k: round(v, 3) for k, v in r.items() if isinstance(v, float)})
            del reg; gc.collect()
    VALORES['barrido'] = filas
    pd.DataFrame(filas).to_csv(DIR_VAL / 'cap08_barrido.csv', index=False)

seccion('barrido', s_barrido)

## 3. Una escena, error por error

In [ ]:
def dibujar(ax, x, x0=0, y0=0, lw=0.6):
    # Aciertos en azul, detecciones sin anotación en naranja y aves anotadas no detectadas en amarillo discontinuo.
    sel = x['conf'] >= CONF_CONTEO
    for c, p in zip(x['cajas'][sel], x['par'][sel]):
        ax.add_patch(Rectangle((c[0] - x0, c[1] - y0), c[2] - c[0], c[3] - c[1], fill=False, ec=ACENTO if p >= 0 else C_FP, lw=lw if p >= 0 else lw * 1.8))
    for g in x['gt'][x['conf_gt'] < CONF_CONTEO]:
        ax.add_patch(Rectangle((g[0] - x0, g[1] - y0), g[2] - g[0], g[3] - g[1], fill=False, ec=C_FN, lw=lw * 1.8, ls=(0, (3, 2))))


def s_escena():
    nombres = [x['archivo'] for x in REF]
    k = nombres.index('C-W1-V0001.jpg') if 'C-W1-V0001.jpg' in nombres else int(np.argmax([len(x['gt']) for x in REF]))
    x = REF[k]; img = leer_rgb(RUTAS[k]); alto, ancho = img.shape[:2]
    sel = x['conf'] >= CONF_CONTEO
    fp = x['cajas'][sel & (x['par'] < 0)]; fn = x['gt'][x['conf_gt'] < CONF_CONTEO]
    centros = np.vstack([(fp[:, :2] + fp[:, 2:]) / 2, (fn[:, :2] + fn[:, 2:]) / 2]) if len(fp) + len(fn) else np.zeros((0, 2))
    lado = int(0.30 * alto); mejor, x0, y0 = -1, 0, 0
    for yy in range(0, alto - lado + 1, lado // 4):
        for xx in range(0, ancho - lado + 1, lado // 4):
            n = int(((centros[:, 0] >= xx) & (centros[:, 0] < xx + lado) & (centros[:, 1] >= yy) & (centros[:, 1] < yy + lado)).sum())
            if n > mejor:
                mejor, x0, y0 = n, xx, yy
    fig = plt.figure(figsize=(ANCHO_IN, 2.85))
    gs = fig.add_gridspec(1, 2, width_ratios=[ancho / alto, 1], wspace=0.04, left=0.005, right=0.995, top=0.92, bottom=0.10)
    a1 = fig.add_subplot(gs[0, 0]); a1.imshow(img); sin_ejes(a1); dibujar(a1, x); a1.set_xlim(0, ancho); a1.set_ylim(alto, 0)
    a1.add_patch(Rectangle((x0, y0), lado, lado, fill=False, ec='white', lw=1.0, ls=(0, (3, 2))))
    a1.set_title('a) Escena completa', loc='left', fontweight='bold')
    a2 = fig.add_subplot(gs[0, 1]); a2.imshow(img[y0:y0 + lado, x0:x0 + lado]); sin_ejes(a2); dibujar(a2, x, x0, y0, lw=1.0)
    a2.set_xlim(0, lado); a2.set_ylim(lado, 0); a2.set_title('b) Detalle', loc='left', fontweight='bold')
    a1.plot([], [], color=ACENTO, lw=1.2, label='Acierto'); a1.plot([], [], color=C_FP, lw=1.6, label='Detección sin anotación')
    a1.plot([], [], color=C_FN, lw=1.6, ls=(0, (3, 2)), label='Ave anotada no detectada')
    fig.legend(*a1.get_legend_handles_labels(), frameon=False, loc='lower center', ncol=3, fontsize=7, bbox_to_anchor=(0.5, 0.0))
    guardar(fig, DIR_FIG / 'Fig_8_4_escena_errores.png'); plt.show()
    VALORES['escena'] = {'archivo': x['archivo'], 'anotadas': int(len(x['gt'])), 'detectadas': int(sel.sum()), 'aciertos': int((sel & (x['par'] >= 0)).sum()),
                         'sin_anotacion': int(len(fp)), 'no_detectadas': int(len(fn)), 'errores_en_el_detalle': int(mejor),
                         'detalle': [int(x0), int(y0), int(lado)], 'ancho': int(ancho), 'alto': int(alto)}
    print(VALORES['escena'])

seccion('escena', s_escena)

## 4. Los errores más claros de cada tipo

In [ ]:
def recorte(img, caja, color, estilo='-'):
    # Recorte cuadrado alrededor de una caja, ampliado a 160 píxeles, con la caja dibujada.
    alto, ancho = img.shape[:2]
    cx, cy = (caja[0] + caja[2]) / 2, (caja[1] + caja[3]) / 2
    lado = int(max(96, 2.4 * max(caja[2] - caja[0], caja[3] - caja[1])))
    x0 = int(np.clip(cx - lado / 2, 0, max(0, ancho - lado))); y0 = int(np.clip(cy - lado / 2, 0, max(0, alto - lado)))
    trozo = img[y0:y0 + lado, x0:x0 + lado]; e = 160 / max(trozo.shape[:2])
    return cv2.resize(trozo, (160, 160), interpolation=cv2.INTER_CUBIC), ((caja[0] - x0) * e, (caja[1] - y0) * e, (caja[2] - caja[0]) * e, (caja[3] - caja[1]) * e)


def s_errores():
    fp, fn = [], []
    for i, x in enumerate(REF):
        sel = (x['conf'] >= CONF_CONTEO) & (x['par'] < 0)
        iou_max = x['M'].max(1) if x['M'].shape[1] else np.zeros(len(x['conf']), np.float32)
        for c, cf, u in zip(x['cajas'][sel], x['conf'][sel], iou_max[sel]):
            fp.append((float(cf), float(u), i, c))
        for g in x['gt'][x['conf_gt'] < CONF_CONTEO]:
            fn.append((float((g[2] - g[0]) * (g[3] - g[1])), i, g))
    iou_fp = np.array([f[1] for f in fp]); conf_fp = np.array([f[0] for f in fp])
    VALORES['errores_detector'] = {'sin_anotacion': len(fp), 'no_detectadas': len(fn),
                                   'sin_anotacion_iou_menor_0_1': int((iou_fp < 0.1).sum()), 'sin_anotacion_iou_0_1_a_0_5': int(((iou_fp >= 0.1) & (iou_fp < 0.5)).sum()),
                                   'sin_anotacion_iou_mayor_0_5': int((iou_fp >= 0.5).sum()), 'sin_anotacion_conf_mayor_0_7': int((conf_fp >= 0.7).sum()),
                                   'sin_anotacion_aisladas_conf_mayor_0_7': int(((conf_fp >= 0.7) & (iou_fp < 0.1)).sum())}
    aisladas = sorted((f for f in fp if f[1] < 0.1), key=lambda f: -f[0])
    elegidas_fp, vistas = [], {}
    for f in aisladas:                       # como mucho dos por imagen, para que la galería sea variada
        if vistas.get(f[2], 0) < 2:
            elegidas_fp.append(f); vistas[f[2]] = vistas.get(f[2], 0) + 1
        if len(elegidas_fp) == 12:
            break
    elegidas_fn, vistas = [], {}
    for f in sorted(fn, key=lambda f: -f[0]):
        if vistas.get(f[1], 0) < 2:
            elegidas_fn.append(f); vistas[f[1]] = vistas.get(f[1], 0) + 1
        if len(elegidas_fn) == 12:
            break
    cache = {}
    def imagen(i):
        if i not in cache:
            cache[i] = leer_rgb(RUTAS[i])
        return cache[i]
    fig = plt.figure(figsize=(ANCHO_IN, 4.9))
    gs = fig.add_gridspec(2, 1, hspace=0.24)
    det = {'sin_anotacion': [], 'no_detectadas': []}
    for b, (titulo, lista, color, estilo) in enumerate((('a) Detecciones sin anotación, las de mayor confianza', elegidas_fp, C_FP, '-'),
                                                        ('b) Aves anotadas no detectadas, las de mayor tamaño', elegidas_fn, C_FN, (0, (3, 2))))):
        sub = gs[b].subgridspec(2, 6, wspace=0.05, hspace=0.20)
        for k in range(12):
            ax = fig.add_subplot(sub[k // 6, k % 6]); sin_ejes(ax)
            if k == 0:
                ax.set_title(titulo, loc='left', fontweight='bold', fontsize=7)
            if k >= len(lista):
                continue
            f = lista[k]; i = f[2] if b == 0 else f[1]; caja = f[3] if b == 0 else f[2]
            trozo, (rx, ry, rw, rh) = recorte(imagen(i), caja, color)
            ax.imshow(trozo); ax.add_patch(Rectangle((rx, ry), rw, rh, fill=False, ec=color, lw=1.2, ls=estilo))
            ax.set_xlim(0, 160); ax.set_ylim(160, 0)
            ax.set_xlabel(f'Confianza {coma(f[0], 2)}' if b == 0 else Path(REF[i]['archivo']).stem, fontsize=5.6, color=TINTA_2, labelpad=2)
            det['sin_anotacion' if b == 0 else 'no_detectadas'].append({'archivo': REF[i]['archivo'], 'caja': [round(float(v), 1) for v in caja],
                                                                         'confianza': round(f[0], 3) if b == 0 else None})
    guardar(fig, DIR_FIG / 'Fig_8_5_galeria_errores.png'); plt.show()
    VALORES['errores_detector']['galeria'] = det
    print({k: v for k, v in VALORES['errores_detector'].items() if k != 'galeria'})

seccion('errores', s_errores)

## 5. Experimento: el mismo detector entrenado a 640 píxeles

El proyecto entrenó a 960 píxeles porque las aves son pequeñas. Aquí se repite el entrenamiento con la misma configuración y un solo cambio, el tamaño de imagen, y se evalúa con el mismo procedimiento.

In [ ]:
def s_640():
    if not ENTRENAR_640:
        print('Entrenamiento a 640 píxeles omitido.'); return
    mejor = DIR_EXP / 'yolo11s_640_best.pt'; rj = DIR_EXP / 'yolo11s_640.json'
    if not (mejor.exists() and rj.exists()):
        random.seed(2026); np.random.seed(2026); torch.manual_seed(2026)
        t0 = time.time()
        YOLO('yolo11s.pt').train(data=str(YAML), epochs=60, imgsz=640, batch=8, device=DISP, workers=4, optimizer='AdamW', lr0=0.002, momentum=0.9,
                                 weight_decay=0.0005, cos_lr=True, patience=15, pretrained=True, amp=True, seed=2026, deterministic=True, close_mosaic=10,
                                 plots=False, save=True, project=str(LOCAL / 'cap08_runs'), name='yolo11s_640', exist_ok=True, verbose=False)
        minutos = (time.time() - t0) / 60
        carpeta = LOCAL / 'cap08_runs' / 'yolo11s_640'
        assert (carpeta / 'weights' / 'best.pt').exists(), 'No se generó best.pt'
        shutil.copy(carpeta / 'weights' / 'best.pt', mejor)
        if (carpeta / 'results.csv').exists():
            shutil.copy(carpeta / 'results.csv', DIR_EXP / 'yolo11s_640_results.csv')
        rj.write_text(json.dumps({'minutos': round(minutos, 1), 'gpu': GPU}), encoding='utf-8')
    else:
        print('REUTILIZADO: entrenamiento a 640 píxeles')
    info = json.loads(rj.read_text(encoding='utf-8'))
    det = YOLO(str(mejor)); filas = []
    for lado in (640, 960):
        reg, ms = pasada(det, lado, 0.7); r = resumir(reg, ms); r['lado_inferencia'] = lado; filas.append(r)
        print('Entrenado a 640, inferencia a', lado, {k: round(v, 3) for k, v in r.items() if isinstance(v, float)})
        if lado == 640:
            pd.DataFrame({'archivo': [x['archivo'] for x in reg], 'anotadas': [len(x['gt']) for x in reg],
                          'detectadas': [int((x['conf'] >= CONF_CONTEO).sum()) for x in reg]}).to_csv(DIR_EXP / 'yolo11s_640_conteo.csv', index=False)
        del reg; gc.collect()
    rcsv = DIR_EXP / 'yolo11s_640_results.csv'
    VALORES['entrenado_640'] = {'minutos': info.get('minutos'), 'epocas': int(len(pd.read_csv(rcsv))) if rcsv.exists() else None, 'evaluacion': filas}

seccion('entrenado_640', s_640)

## 6. Guardar y descargar

In [ ]:
VALORES['errores'] = ERRORES
VALORES['entorno'] = {'gpu': GPU, 'torch': torch.__version__, 'ultralytics': ultralytics.__version__}
ruta_json = DIR_VAL / 'valores_cap08.json'
ruta_json.write_text(json.dumps(VALORES, ensure_ascii=False, indent=1, default=str), encoding='utf-8')
ruta_zip = LOCAL / 'AgroVision_cap08.zip'
with zipfile.ZipFile(ruta_zip, 'w', zipfile.ZIP_DEFLATED) as z:
    for f in sorted(DIR_FIG.glob('Fig_8_*.png')):
        z.write(f, 'cap08/' + f.name)
    for f in [ruta_json, DIR_VAL / 'cap08_detecciones.npz', DIR_VAL / 'cap08_imagenes.csv', DIR_VAL / 'cap08_barrido.csv'] + sorted(DIR_EXP.glob('*.csv')) + sorted(DIR_EXP.glob('*.json')):
        if f.is_file():
            z.write(f, f.name)
print('Secciones con error:', list(ERRORES) or 'ninguna')
print('Figuras:', len(FIGURAS), '· zip de', round(ruta_zip.stat().st_size / 1e6, 1), 'MB')
if EN_COLAB:
    from google.colab import files
    files.download(str(ruta_zip))